# Baseline evaluation (saved `best.pt`)
Run from the repository root or `main_notebooks/`. This notebook never retrains or overwrites the baseline. Validation is used for model selection; it is not an independent test set.

**Primary measure:** one alarm per physical double tap on a continuous recording. An alarm emitted **2 or more frames before** the labeled second tap is a false positive; the tap is a false negative unless another alarm detects it. An alarm 1 frame early is allowed for annotation jitter. Late matches are allowed up to 30 frames (0.3 s at 100 Hz). Timing is measured at *emission*, not at the probability peak.

The repository's `model.step()` is not equivalent to batch `forward()`. Full-recording `forward()` is causal in eval mode; we check it against the corrected chunked causal implementation. Postprocessing consumes only probabilities available at each emission frame. This assesses the learned model's causal behavior, but not the current buggy deployed `step()` implementation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Open from repository root or main_notebooks/'
sys.path.insert(0, str(ROOT))

from tap_recognition.config import TrainConfig
from tap_recognition.model import CausalCNNGRU
from train import build_datasets, evaluate, window_class_from_probs
from notebooks_analysis.state_reset_analysis import load_recordings, overlap_stream_logits

# Change this when evaluating a different run.
RUN_DIR = ROOT / 'testing_checkpoints/20260925_112046'
checkpoint = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=True)
saved = checkpoint['train_config']
cfg = TrainConfig()
for key, value in saved['data'].items():
    setattr(cfg.data, key, value)
for key, value in saved['labels'].items():
    setattr(cfg.labels, key, value)
for key, value in saved['training'].items():
    if key == 'early_stopping':
        for name, setting in value.items():
            setattr(cfg.training.early_stopping, name, setting)
    else:
        setattr(cfg.training, key, value)
cfg.seed = saved['seed']
for key in ('train_dir', 'val_dir'):
    setattr(cfg.data, key, ','.join(str(ROOT / p.strip()) for p in getattr(cfg.data, key).split(',')))
model_cfg = dict(checkpoint['model_config'])
model_cfg['dilations'] = tuple(model_cfg['dilations'])
model = CausalCNNGRU(**model_cfg).cpu().eval()
model.load_state_dict(checkpoint['model_state'])
torch.set_num_threads(2)
run_info = pd.DataFrame([dict(run=str(RUN_DIR), epoch=checkpoint['epoch'], seed=cfg.seed,
                             threshold=cfg.training.prediction_threshold,
                             torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__)])
run_info

## 1. Training history: underfitting / overfitting
The training notebook must have completed a fresh run to write `history.csv`. The old notebook output alone is not a trustworthy replacement for a saved history.

In [ ]:
history_path = RUN_DIR / 'history.csv'
history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
if history.empty:
    print('No history.csv in this run. Re-run baseline.ipynb from a fresh kernel and point RUN_DIR at that new run.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, metric in zip(axes[:2], ('loss', 'window_acc')):
        for split in ('train', 'val'):
            ax.plot(history.epoch, history[f'{split}_{metric}'], label=split)
        ax.axvline(checkpoint['epoch'], color='gray', linestyle=':', label='best checkpoint')
        ax.set(xlabel='Epoch', ylabel=metric, title=metric)
        ax.legend()
    if {'val_precision', 'val_recall'} <= set(history.columns):
        axes[2].plot(history.epoch, history.val_precision, label='val precision')
        axes[2].plot(history.epoch, history.val_recall, label='val recall')
        if {'val_tp', 'val_fp', 'val_fn'} <= set(history.columns):
            history['val_f1'] = 2*history.val_tp / (2*history.val_tp + history.val_fp + history.val_fn)
            axes[2].plot(history.epoch, history.val_f1, label='val F1')
        axes[2].axvline(checkpoint['epoch'], color='gray', linestyle=':')
        axes[2].set(xlabel='Epoch', ylabel='Score', title='Validation event vs none (windows)')
        axes[2].legend()
    if 'lr' in history:
        print('Learning-rate schedule is available in history.lr')
    plt.tight_layout()
    plt.show()

In [ ]:
history  # Filter by epoch; compare train_loss / val_loss and train_window_acc / val_window_acc.

## 2. Window-level diagnostics
Each observation is a 300-frame training/validation window. PR-AUC uses one event score per window: the maximum probability of either event class across the window. These are *not* alarm metrics.

In [ ]:
def divide(a, b):
    return float(a / b) if b else float('nan')

def ranking_metrics(truth, score):
    # Average precision = area under the stepwise precision-recall curve.
    truth, score = np.asarray(truth, dtype=bool), np.asarray(score)
    positives, negatives = truth.sum(), (~truth).sum()
    if not positives or not negatives:
        return dict(pr_auc=float('nan'), roc_auc=float('nan'))
    order = np.argsort(-score, kind='stable')
    y, s = truth[order], score[order]
    ends = np.r_[np.flatnonzero(s[:-1] != s[1:]), len(s) - 1]
    tp = np.cumsum(y)[ends]
    precision = tp / (ends + 1)
    pr_auc = np.sum(precision * np.diff(np.r_[0, tp / positives]))
    ranks = rankdata(score, method='average')
    roc_auc = (ranks[truth].sum() - positives * (positives + 1) / 2) / (positives * negatives)
    return dict(pr_auc=float(pr_auc), roc_auc=float(roc_auc))

def binary_metrics(truth, predicted):
    truth, predicted = np.asarray(truth, dtype=bool), np.asarray(predicted, dtype=bool)
    tp, fp = int((truth & predicted).sum()), int((~truth & predicted).sum())
    fn, tn = int((truth & ~predicted).sum()), int((~truth & ~predicted).sum())
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=divide(tp, tp+fp),
                recall=divide(tp, tp+fn), f1=divide(2*tp, 2*tp+fp+fn),
                specificity=divide(tn, tn+fp), accuracy=divide(tp+tn, len(truth)),
                balanced_acc=np.nanmean([divide(tp, tp+fn), divide(tn, tn+fp)]))

_, val_ds = build_datasets(cfg)
val_loader = DataLoader(val_ds, batch_size=cfg.training.batch_size, shuffle=False)
saved_metrics = checkpoint['val_metrics']
measured_metrics = evaluate(model, val_loader, torch.device('cpu'),
                            cfg.training.prediction_threshold, cfg.training.event_loss_weight,
                            cfg.training.neg_window_weight)
for key in ('loss', 'window_acc', 'precision', 'recall', 'tp', 'fp', 'fn', 'tn'):
    assert np.isclose(saved_metrics[key], measured_metrics[key], atol=1e-6), key
print('Saved validation metrics independently reproduced.')
window_rows = []
with torch.inference_mode():
    for index in range(len(val_ds)):
        sample = val_ds[index]
        probs = model(sample['imu'].unsqueeze(0))[0].softmax(-1)
        event = probs[0, :, 1:]
        score = event.max().item()
        predicted = int(window_class_from_probs(probs, cfg.training.prediction_threshold)[0])
        window_rows.append(dict(index=index, file=val_ds.window_meta[index].source_file,
                                true=int(sample['window_label']), pred=predicted,
                                event_score=score, peak_local_frame=int(event.max(-1).values.argmax()),
                                true_global_frame=val_ds.window_meta[index].trigger_frame))
windows = pd.DataFrame(window_rows)
window_summary = pd.DataFrame([binary_metrics(windows.true > 0, windows.pred > 0) |
                               ranking_metrics(windows.true > 0, windows.event_score) |
                               dict(window_3class_accuracy=float((windows.true == windows.pred).mean()))])
window_summary

In [ ]:
window_cm = pd.crosstab(windows.true, windows.pred, rownames=['true'], colnames=['predicted'])
window_cm = window_cm.reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)
per_class = pd.DataFrame([dict(class_id=c, precision=divide(window_cm.loc[c, c], window_cm[c].sum()),
                               recall=divide(window_cm.loc[c, c], window_cm.loc[c].sum()),
                               f1=divide(2*window_cm.loc[c, c], window_cm[c].sum()+window_cm.loc[c].sum()),
                               support=int(window_cm.loc[c].sum())) for c in (0, 1, 2)])
correct_positive = windows[(windows.true > 0) & (windows.pred > 0)]
print('3-class confusion matrix (rows=true, columns=predicted)')
display(window_cm)
print('Side accuracy given an event was detected:',
      divide((correct_positive.true == correct_positive.pred).sum(), len(correct_positive)))
per_class

In [ ]:
windows  # Filter file, true, pred, event_score and peak_local_frame to inspect individual errors.

## 3. Whole-recording causal frame probabilities
The model state persists within a recording, resets only between recordings, and the high-pass filter is applied once per recording. Frame targets are *hard regions* around each second tap (±10 frames), distinct from the soft labels used for training. Frame accuracy is dominated by background and remains secondary.

In [ ]:
all_recordings, recording_audit, _ = load_recordings(ROOT, saved['data'], cfg.labels.half_frames)
recordings = [r for r in all_recordings if r.split == 'valid']
assert len(recordings) > 0
probabilities, frame_parts = {}, []
with torch.inference_mode():
    for rec in recordings:
        x = torch.from_numpy(rec.imu).unsqueeze(0)
        logits, _ = model(x)
        # Independent causal reference: only new CNN features advance the GRU.
        sizes = [min(128, x.shape[1] - i) for i in range(0, x.shape[1], 128)]
        torch.testing.assert_close(overlap_stream_logits(model, x, sizes), logits, atol=2e-5, rtol=2e-5)
        p = logits[0].softmax(-1).numpy()
        probabilities[rec.path.name] = p
        target = np.zeros(len(p), dtype=np.int8)
        for frame, _ in rec.events:
            target[max(0, frame-10):min(len(p), frame+11)] = 1
        frame_parts.append(pd.DataFrame(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                             frame=np.arange(len(p)), true=target,
                                             p_none=p[:, 0], p_left=p[:, 1], p_right=p[:, 2],
                                             event_score=p[:, 1:].max(axis=1))))
frames = pd.concat(frame_parts, ignore_index=True)
frame_summary = pd.DataFrame([binary_metrics(frames.true, frames.event_score >= cfg.training.prediction_threshold) |
                              ranking_metrics(frames.true, frames.event_score)])
frame_summary

In [ ]:
frames[frames['event_score'] > 0.8]  # Each row is a real recorded frame; filter file and frame range for detailed inspection.

## 4. Streaming alarms and one-to-one event matching
Two *predeclared* policies: `instant` emits on the first crossing (no look-ahead); `lookahead_12f` waits 12 frames like the current default detector. Both require one frame above threshold and enforce a 0.5 s refractory period after emission. This comparison makes the latency/precision tradeoff visible. A fixed 30-frame late matching limit is used for primary event F1; stricter deadlines are analyzed next. Do not tune a threshold on validation and then call that same validation performance an unbiased test result.

In [ ]:
THRESHOLDS = np.round(np.arange(0.1, 1.0, 0.1), 2)  # includes the training threshold 0.50
POLICIES = {'instant': 0, 'lookahead_12f': 12}
MAX_LATE_FRAMES = 30
EARLY_ALLOWANCE_FRAMES = 1  # delta=-2 is never a match

def emit_alarms(probs, threshold, look, refractory):
    # All decisions use information available at the emission frame, never future frames.
    alarms, i, n = [], 0, len(probs)
    while i < n:
        score = probs[i, 1:].max()
        if score < threshold:
            i += 1
            continue
        emission = i + look
        if emission >= n:  # An offline end-of-file flush is not an online alarm.
            break
        peak = i + int(probs[i:emission+1, 1:].max(axis=1).argmax())
        cls = int(probs[peak, 1:].argmax() + 1)
        alarms.append(dict(alarm_frame=emission, peak_frame=peak, class_id=cls,
                           probability=float(probs[peak, cls])))
        # Match the detector's lockout: it is measured from the selected peak.
        i = max(emission + 1, peak + max(1, refractory))
    return alarms

def match_alarms(truth, alarms, class_aware=True):
    # Min-cost 1:1 assignment: prefer matching as many events as possible, then closest timing.
    n, m = len(truth), len(alarms)
    if not n or not m:
        return []
    cost = np.full((n, m+n), 1e6, dtype=float)
    cost[:, m:] = 1e3
    for a, (frame, cls) in enumerate(truth):
        for b, alarm in enumerate(alarms):
            delta = alarm['alarm_frame'] - frame
            if -EARLY_ALLOWANCE_FRAMES <= delta <= MAX_LATE_FRAMES and (not class_aware or cls == alarm['class_id']):
                cost[a, b] = abs(delta)
    rows, cols = linear_sum_assignment(cost)
    return [(int(a), int(b)) for a, b in zip(rows, cols) if b < m and cost[a, b] < 1e3]

assert not match_alarms([(100, 1)], [dict(alarm_frame=98, class_id=1)])
assert len(match_alarms([(100, 1)], [dict(alarm_frame=99, class_id=1)])) == 1
assert not match_alarms([(100, 1)], [dict(alarm_frame=100, class_id=2)])
assert len(match_alarms([(100, 1)], [dict(alarm_frame=98, class_id=1),
                                   dict(alarm_frame=101, class_id=1)])) == 1
assert emit_alarms(np.array([[0., 1., 0.], [0., 1., 0.]]), .5, 0, 5)[0]['alarm_frame'] == 0

session_rows, alarm_rows, truth_rows = [], [], []
for rec in recordings:
    probs = probabilities[rec.path.name]
    minutes = len(probs) / rec.fs / 60
    for policy, look in POLICIES.items():
        for threshold in THRESHOLDS:
            threshold = float(threshold)
            detected = emit_alarms(probs, threshold, look, round(.5 * rec.fs))
            matches = match_alarms(rec.events, detected)
            binary_matches = match_alarms(rec.events, detected, class_aware=False)
            matched_truth = {a: b for a, b in matches}
            matched_alarms = {b: a for a, b in matches}
            session_rows.append(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                     policy=policy, threshold=threshold, minutes=minutes,
                                     n_true=len(rec.events), n_alarms=len(detected),
                                     tp=len(matches), fp=len(detected)-len(matches),
                                     fn=len(rec.events)-len(matches),
                                     binary_tp=len(binary_matches),
                                     binary_fp=len(detected)-len(binary_matches),
                                     binary_fn=len(rec.events)-len(binary_matches)))
            for a, (frame, cls) in enumerate(rec.events):
                alarm = detected[matched_truth[a]] if a in matched_truth else None
                truth_rows.append(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                       policy=policy, threshold=threshold, true_frame=frame, true_class=cls,
                                       detected=alarm is not None, alarm_frame=alarm['alarm_frame'] if alarm else np.nan,
                                       latency_frames=alarm['alarm_frame']-frame if alarm else np.nan,
                                       fs=rec.fs))
            for b, alarm in enumerate(detected):
                a = matched_alarms.get(b)
                closest = min(rec.events, key=lambda event: abs(alarm['alarm_frame']-event[0])) if rec.events else None
                nearest_delta = alarm['alarm_frame'] - closest[0] if closest else np.nan
                alarm_rows.append(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                       policy=policy, threshold=threshold, **alarm,
                                       matched=a is not None, true_frame=rec.events[a][0] if a is not None else np.nan,
                                       latency_frames=alarm['alarm_frame']-rec.events[a][0] if a is not None else np.nan,
                                       nearest_true_frame=closest[0] if closest else np.nan,
                                       delta_to_nearest_true=nearest_delta,
                                       too_early=bool(-MAX_LATE_FRAMES <= nearest_delta <= -2) if closest else False))
sessions = pd.DataFrame(session_rows)
alarms = pd.DataFrame(alarm_rows)
truth_events = pd.DataFrame(truth_rows)
print('Validation recordings:', len(recordings), '| positive events:', len([e for r in recordings for e in r.events]))

sessions[(sessions['threshold'] == 0.5) & (sessions['policy'] == 'lookahead_12f')]

In [ ]:
event_rows = []
for (policy, threshold), group in sessions.groupby(['policy', 'threshold'], sort=True):
    tp, fp, fn = (int(group[col].sum()) for col in ('tp', 'fp', 'fn'))
    bt, bp, bn = (int(group[col].sum()) for col in ('binary_tp', 'binary_fp', 'binary_fn'))
    negative = group[group.n_true == 0]
    event_rows.append(dict(policy=policy, threshold=threshold, tp=tp, fp=fp, fn=fn,
                           precision=divide(tp, tp+fp), recall=divide(tp, tp+fn),
                           f1=divide(2*tp, 2*tp+fp+fn),
                           binary_f1=divide(2*bt, 2*bt+bp+bn),
                           fp_per_min=divide(fp, group.minutes.sum()),
                           negative_fp_per_min=divide(negative.fp.sum(), negative.minutes.sum())))
event_summary = pd.DataFrame(event_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for policy, group in event_summary.groupby('policy'):
    axes[0].plot(group.recall, group.precision, 'o-', label=policy)
    axes[1].plot(group.threshold, group.negative_fp_per_min, 'o-', label=policy)
axes[0].set(xlabel='Event recall', ylabel='Event precision', title='Class-aware event PR (threshold sweep)')
axes[1].set(xlabel='Probability threshold', ylabel='False alarms/min on negative-only recordings')
for ax in axes: ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
event_summary  # Compare threshold, instant vs lookahead, class-aware F1, binary F1 and FP/min.

In [ ]:
sessions  # Filter threshold, policy, user, action or file to see where errors occur.

In [ ]:
alarms[(~alarms['true_frame'].isna()) & (alarms['policy'] == 'instant') & (abs(alarms['latency_frames']) > 5)]  # Filter too_early == True, threshold, policy, file or matched == False.

In [ ]:
alarms[alarms.too_early]  # Alarms 2–30 frames before a labeled second tap; none are matched TPs.

In [ ]:
class_rows = []
for (policy, threshold), positives in truth_events.groupby(['policy', 'threshold']):
    predictions = alarms[(alarms.policy == policy) & (alarms.threshold == threshold)]
    for cls in (1, 2):
        target = positives[positives.true_class == cls]
        classified = predictions[predictions.class_id == cls]
        tp = int(target.detected.sum())
        fp, fn = len(classified)-tp, len(target)-tp
        class_rows.append(dict(policy=policy, threshold=threshold, class_id=cls,
                               support=len(target), tp=tp, fp=fp, fn=fn,
                               precision=divide(tp, tp+fp), recall=divide(tp, tp+fn),
                               f1=divide(2*tp, 2*tp+fp+fn)))
event_per_class = pd.DataFrame(class_rows)
event_per_class  # Per-class detection performance, including wrong-side alarms as FP + FN.

## 5. Latency and recall before a deadline
Latency is `alarm_frame - labeled_second_tap_frame`; negative values are early. The primary event score accepts at most one frame early, but an alarm 2 frames early cannot be a TP. Deadline recall has **all annotated events** as denominator, including misses. Strict `<5` means 0–4 frames after the tap (plus the allowed one-frame annotation jitter). At 100 Hz, 5 frames = 50 ms.

In [ ]:
DEADLINES = (5, 10, 15, 20, 30)
latency_rows = []
for (policy, threshold), group in truth_events.groupby(['policy', 'threshold']):
    matched = group[group.detected]
    lat = matched.latency_frames.to_numpy()
    row = dict(policy=policy, threshold=threshold, annotated=len(group), matched=len(matched),
               mean_frames=np.mean(lat) if len(lat) else np.nan,
               median_frames=np.median(lat) if len(lat) else np.nan,
               p90_frames=np.percentile(lat, 90) if len(lat) else np.nan,
               mean_ms=np.mean(lat / matched.fs.to_numpy() * 1000) if len(lat) else np.nan,
               early_1_frame=int((lat == -1).sum()))
    for deadline in DEADLINES:
        row[f'recall_lt_{deadline}f'] = divide(int((lat < deadline).sum()), len(group))
    latency_rows.append(row)
latency_summary = pd.DataFrame(latency_rows)
at_default = truth_events[(truth_events.policy == 'instant') &
                          (truth_events.threshold == cfg.training.prediction_threshold)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(at_default.loc[at_default.detected, 'latency_frames'], bins=np.arange(-2, 33, 2))
axes[0].set(xlabel='Alarm latency (frames)', ylabel='Matched events', title='Instant policy @ threshold 0.5')
for policy, group in latency_summary.groupby('policy'):
    axes[1].plot(group.threshold, group['recall_lt_5f'], 'o-', label=f'{policy}: <5 frames')
    axes[1].plot(group.threshold, group['recall_lt_10f'], '--', label=f'{policy}: <10 frames')
axes[1].set(xlabel='Threshold', ylabel='Fraction of all annotated taps', ylim=(0, 1))
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
latency_summary  # Filter threshold/policy; mean latency is conditional on matched events only.

In [ ]:
truth_events  # Includes missed events (detected=False); filter latency_frames and user/action/file.

## 6. Inspect one recording
Set `FILE` and `POLICY` below, then zoom with `START_FRAME` / `END_FRAME`. Red: annotated second tap. Green: actual alarm emission. Orange: probability peak selected for the alarm (may differ from emission). An early alarm 2+ frames before a tap is **not** shown as a matched detection.

In [ ]:
FILE = next(r.path.name for r in recordings if r.events)  # or choose any file in sessions.file
POLICY = 'instant'
THRESHOLD = 0.5
rec = next(r for r in recordings if r.path.name == FILE)
START_FRAME, END_FRAME = 0, len(rec.imu)  # e.g. 100, 450
p = probabilities[FILE]
selected = alarms[(alarms.file == FILE) & (alarms.policy == POLICY) &
                  (alarms.threshold == THRESHOLD)].copy()
x = np.arange(START_FRAME, END_FRAME) / rec.fs
fig, (ax, ap) = plt.subplots(2, 1, figsize=(45, 6), sharex=True)
ax.plot(x, np.linalg.norm(rec.imu[START_FRAME:END_FRAME, :3], axis=1), label='HP acceleration magnitude')
ax.set(ylabel='Magnitude', title=FILE)
ap.plot(x, p[START_FRAME:END_FRAME, 1], label='P(left)', alpha=.8)
ap.plot(x, p[START_FRAME:END_FRAME, 2], label='P(right)', alpha=.8)
ap.axhline(THRESHOLD, color='gray', linestyle=':', label='Threshold')
for frame, _ in rec.events:
    if START_FRAME <= frame < END_FRAME:
        for a in (ax, ap): a.axvline(frame/rec.fs, color='red', alpha=.6, label='Labeled second tap')
for row in selected.itertuples():
    if START_FRAME <= row.alarm_frame < END_FRAME:
        for a in (ax, ap): a.axvline(row.alarm_frame/rec.fs, color='green', alpha=.7, label='Alarm emission')
    if START_FRAME <= row.peak_frame < END_FRAME:
        ap.axvline(row.peak_frame/rec.fs, color='orange', alpha=.5, linestyle=':', label='Selected peak')
ap.set(xlabel='Time (s)', ylabel='Probability', ylim=(-.05, 1.05))
for a in (ax, ap):
    handles, labels = a.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    a.legend(unique.values(), unique.keys(), loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
selected  # The exact alarms drawn above; filter matched, alarm_frame, peak_frame or latency_frames.

## 7. All metrics in one place
`metrics_summary` is the complete numeric registry, including every evaluated threshold and policy. Window and frame metrics refer to different sample units, so their precision/recall should not be compared directly with event metrics. `default_summary` is a compact overview at threshold 0.50.

In [ ]:
summary_parts = []
def collect_metrics(table, level, fixed=None):
    fixed = fixed or {}
    keys = [key for key in ('policy', 'threshold', 'class_id') if key in table.columns]
    return table.melt(id_vars=keys, var_name='metric', value_name='value').assign(level=level, **fixed)

summary_parts.append(collect_metrics(window_summary, 'window'))
summary_parts.append(collect_metrics(per_class, 'window_class'))
summary_parts.append(collect_metrics(frame_summary, 'frame'))
summary_parts.append(collect_metrics(event_summary, 'event'))
summary_parts.append(collect_metrics(event_per_class, 'event_class'))
summary_parts.append(collect_metrics(latency_summary, 'latency'))
summary_parts.append(collect_metrics(pd.DataFrame([{'epoch': checkpoint['epoch'], **saved_metrics}]), 'saved_checkpoint'))
cm_counts = pd.DataFrame([{'metric': f'window_cm_true_{t}_pred_{p}',
                           'value': int(window_cm.loc[t, p])} for t in range(3) for p in range(3)])
summary_parts.append(cm_counts.assign(level='window_confusion'))
if not history.empty:
    summary_parts.append(collect_metrics(history, 'epoch_history'))
metrics_summary = pd.concat(summary_parts, ignore_index=True)
metrics_summary = metrics_summary[['level', 'policy', 'threshold', 'class_id', 'metric', 'value']]
default_summary = pd.DataFrame([
    dict(level='window', policy='independent windows', threshold=cfg.training.prediction_threshold,
         precision=window_summary.precision.iloc[0], recall=window_summary.recall.iloc[0],
         f1=window_summary.f1.iloc[0], pr_auc=window_summary.pr_auc.iloc[0],
         false_alarms_per_min=np.nan, recall_lt_5f=np.nan, mean_latency_ms=np.nan),
    dict(level='frame', policy='continuous causal probabilities', threshold=cfg.training.prediction_threshold,
         precision=frame_summary.precision.iloc[0], recall=frame_summary.recall.iloc[0],
         f1=frame_summary.f1.iloc[0], pr_auc=frame_summary.pr_auc.iloc[0],
         false_alarms_per_min=np.nan, recall_lt_5f=np.nan, mean_latency_ms=np.nan),
] + [dict(level='event', policy=row.policy, threshold=row.threshold, precision=row.precision,
          recall=row.recall, f1=row.f1, pr_auc=np.nan, false_alarms_per_min=row.negative_fp_per_min,
          recall_lt_5f=float(latency_summary.loc[(latency_summary.policy == row.policy) &
                                      (latency_summary.threshold == row.threshold), 'recall_lt_5f'].iloc[0]),
          mean_latency_ms=float(latency_summary.loc[(latency_summary.policy == row.policy) &
                                    (latency_summary.threshold == row.threshold), 'mean_ms'].iloc[0]))
     for row in event_summary[event_summary.threshold == cfg.training.prediction_threshold].itertuples()])
default_summary

In [ ]:
metrics_summary  # Filter level, policy, threshold, class_id or metric to see all calculated numbers.